# Story 3: A Parameter Study with Many Short Calculations
**Who:** A researcher screening a grid of strains and temperatures. Each calculation takes seconds to minutes, but
there are hundreds of them.

**What they want:** Run the whole grid in one SLURM allocation, find out which parameter sets fail, and collect the
results in a table.

**Why plain SLURM is painful:** One `sbatch` job per calculation floods the queue, every job waits separately, and many
clusters limit the number of jobs per user. Job arrays require mapping an index to parameters and collecting the
results from files. A hand-written loop of `srun ... &` calls in one batch script needs manual bookkeeping of free
cores, and the one failing parameter set is buried in hundreds of log files.

**How executorlib helps:** The `SlurmJobExecutor` runs inside an existing allocation and distributes the function calls
over its cores as `srun` job steps - no new jobs enter the queue. Errors are collected in a single log file and
results are cached, so they can be loaded into a [pandas](https://pandas.pydata.org) DataFrame.

This notebook has to run **inside a SLURM allocation** with at least two cores, for example started with
`salloc -n 2` or from a batch script submitted with `sbatch -n 2`.

## Cluster-specific settings
**Adapt the following cell to your cluster and allocation.** The values below are the ones used to test this notebook in a two-core allocation of a single-node SLURM test cluster with MPICH. For OpenMPI >= 5 use `["--mpi=pmix"]`; set `max_cores` to the number of cores of your allocation.

In [ ]:
# Cluster-specific settings - adapt these values to your cluster and allocation.
slurm_cmd_args = ["--mpi=pmi2"]  # additional arguments for srun, e.g. the MPI plugin
max_cores = 2  # number of CPU cores of the allocation executorlib may use

## Step 1: Write the function
The toy model fails for large strains at high temperature, just like a real simulation which does not converge for
some parameter sets.

In [ ]:
import time


def compute_energy(strain, temperature):
    # Stand-in for a short simulation (seconds to minutes), e.g. the energy of a strained crystal.
    time.sleep(0.5)
    if strain > 0.04 and temperature > 600:
        raise ValueError("simulation did not converge")
    return 100.0 * strain**2 * (1.0 - temperature / 3000.0)


strain_lst = [0.0, 0.01, 0.02, 0.03, 0.05]
temperature_lst = [300, 600, 900]
parameter_lst = [(strain, temperature) for strain in strain_lst for temperature in temperature_lst]
len(parameter_lst)

## Step 2: Run the grid and find the failing parameter sets
`max_cores` limits how many cores executorlib uses at the same time; with one core per call it executes `max_cores`
calculations in parallel and starts the next one as soon as a core becomes free. The `error_log_file` in the
`resource_dict` collects the traceback of every failing call together with its function and arguments in one file -
use an absolute path, as the calculations do not necessarily run in the current directory.

In [ ]:
import os
from executorlib import SlurmJobExecutor

cache_directory = os.path.abspath("story3_cache")
error_log_file = os.path.abspath("story3_errors.log")

with SlurmJobExecutor(
    max_cores=max_cores,
    cache_directory=cache_directory,
    resource_dict={"cores": 1, "slurm_cmd_args": slurm_cmd_args, "error_log_file": error_log_file},
) as exe:
    future_lst = [exe.submit(compute_energy, strain, temperature) for strain, temperature in parameter_lst]
    failed_lst = [p for p, future in zip(parameter_lst, future_lst) if future.exception() is not None]

failed_lst

In [ ]:
with open(error_log_file) as f:
    print(f.read())

## Step 3: Collect the results in a table
With a `cache_directory` every successful result is stored together with its input arguments and run time.
`get_cache_data()` returns them as a list of dictionaries which converts directly into a DataFrame. Failed calls are not
cached by the `SlurmJobExecutor`, so they are simply executed again on the next run.

In [ ]:
import pandas
from executorlib import get_cache_data

df = pandas.DataFrame(get_cache_data(cache_directory=cache_directory))
df["strain"] = df["input_args"].apply(lambda args: args[0])
df["temperature"] = df["input_args"].apply(lambda args: args[1])
df[["strain", "temperature", "output", "runtime"]].sort_values(["temperature", "strain"])

## Step 4: Extend the study with `map()`
Your supervisor asks for two more strains. The `map()` function of the standard library
[Executor interface](https://docs.python.org/3/library/concurrent.futures.html#concurrent.futures.Executor.map) is the
most compact way to apply one function to many parameters. Using the same `cache_directory`, the results computed
above are loaded from the cache and only the new parameter sets are calculated. `map()` raises the first exception it
encounters, so the known failing parameter sets are excluded.

In [ ]:
new_parameter_lst = [(0.015, 300), (0.025, 300)]
valid_lst = [p for p in parameter_lst + new_parameter_lst if p not in failed_lst]

with SlurmJobExecutor(
    max_cores=max_cores,
    cache_directory=cache_directory,
    resource_dict={"cores": 1, "slurm_cmd_args": slurm_cmd_args},
) as exe:
    energy_lst = list(exe.map(compute_energy, [s for s, _ in valid_lst], [t for _, t in valid_lst]))

len(energy_lst), len(get_cache_data(cache_directory=cache_directory))

## Step 5: Post-process results in batches as they arrive
Some follow-up steps are cheaper in groups, for example writing results to a database or fitting a model to a few
data points at a time. `batched()` groups futures into batches of `n` in the order in which they *finish*, so the first
batch is processed while other calculations are still running.

In [ ]:
def summarize(energies):
    # Stand-in for a follow-up step working on a group of results, e.g. storing them in a database.
    return {"n": len(energies), "max_energy": max(energies)}


with SlurmJobExecutor(
    max_cores=max_cores,
    cache_directory=cache_directory,
    resource_dict={"cores": 1, "slurm_cmd_args": slurm_cmd_args},
) as exe:
    future_lst = [exe.submit(compute_energy, strain, temperature) for strain, temperature in valid_lst]
    summary_lst = [exe.submit(summarize, batch) for batch in exe.batched(future_lst, n=4)]
    print([summary.result() for summary in summary_lst])

## Thousands of tasks: SLURM with flux
Every call of the `SlurmJobExecutor` creates a SLURM job step, and every job step is registered in the central SLURM
database. For thousands of short calculations start the hierarchical [flux](https://flux-framework.org) scheduler
inside the allocation with `srun flux start python my_study.py` and replace the `SlurmJobExecutor` by the
`FluxJobExecutor` - the rest of the code stays the same. See
[SLURM with flux](https://executorlib.readthedocs.io/en/latest/3-hpc-job.html#slurm-with-flux).

## Clean up

In [ ]:
import shutil

shutil.rmtree(cache_directory, ignore_errors=True)
if os.path.exists(error_log_file):
    os.remove(error_log_file)

## What you learned
* The `SlurmJobExecutor` distributes many function calls over one allocation; `max_cores` limits the cores in use.
* `error_log_file` collects the failing calls with their arguments in one file.
* `cache_directory` plus `get_cache_data()` turns the results into a DataFrame and skips finished calculations when
  the study is extended with `map()`.
* `batched()` groups results by completion order for follow-up steps.

**Reference:** [HPC Job Executor](https://executorlib.readthedocs.io/en/latest/3-hpc-job.html),
[write log files](https://executorlib.readthedocs.io/en/latest/1-single-node.html#write-log-files), [cache](https://executorlib.readthedocs.io/en/latest/1-single-node.html#cache) and
[runtime-dependent batching](https://executorlib.readthedocs.io/en/latest/1-single-node.html#runtime-dependent-batching).